# End of week 1 exercise

To demonstrate your familiarity with OpenAI API, and also Ollama, build a tool that takes a technical question,  
and responds with an explanation. This is a tool that you will be able to use yourself during the course!

In [1]:
import os
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from openai import OpenAI

# 1. Load your credentials
load_dotenv()

# 2. Configure the OpenRouter Client
# OpenRouter requires /api/v1 at the end of the base_url
client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.getenv("OPENROUTER_API_KEY"),
)

# 3. Define the question and prompts
question = """
Explain the difference between a list comprehension and a generator expression in Python. 
When should I use one over the other?
"""

system_prompt = "You are a helpful technical tutor who answers questions about software engineering, Python, and data science."
user_prompt = f"Please provide a detailed explanation for this question: {question}"

messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": user_prompt}
]

# 4. Get the answer with streaming
print("Tutor is generating your explanation...\n")

try:
    # Using a fast, reliable model from OpenRouter
    stream = client.chat.completions.create(
        model="google/gemini-2.0-flash-001", 
        messages=messages,
        stream=True,
        extra_headers={
            "HTTP-Referer": "http://localhost:3000", # Optional for OpenRouter
            "X-Title": "Technical Tutor Tool",       # Optional for OpenRouter
        }
    )

    response = ""
    display_handle = display(Markdown(""), display_id=True)

    for chunk in stream:
    # Safely check if the chunk has choices and if those choices have a delta
      if chunk.choices and chunk.choices[0].delta.content is not None:
        content = chunk.choices[0].delta.content
        response += content
        # Update the display
        update_display(Markdown(response), display_id=display_handle.display_id)


except Exception as e:
    print(f"❌ Error: {e}")


Tutor is generating your explanation...



Okay, I can help you with a detailed explanation of the differences between list comprehensions and generator expressions in Python, along with guidance on when to choose one over the other.

**List Comprehensions**

*   **Definition:** A concise way to create a new list based on an existing iterable (e.g., a list, tuple, range).
*   **Syntax:** `[expression for item in iterable if condition]`
*   **Evaluation:**  List comprehensions are *eagerly* evaluated. This means that when you execute a list comprehension, the entire list is created and stored in memory immediately.
*   **Memory Usage:** Consumes memory proportional to the size of the resulting list. If the list is very large, this can lead to significant memory usage.
*   **Return Value:** A new `list` object.
*   **Example:**

    ```python
    numbers = [1, 2, 3, 4, 5]
    squares = [x*x for x in numbers]  # squares will be [1, 4, 9, 16, 25]
    ```

**Generator Expressions**

*   **Definition:** Similar to list comprehensions but create a generator iterator. A generator is a special type of iterator that generates values on demand (lazily).
*   **Syntax:** `(expression for item in iterable if condition)`  Note the parentheses instead of square brackets.
*   **Evaluation:** Generator expressions are *lazily* evaluated.  They do not compute and store all the values at once. Instead, they produce each value only when it is requested (e.g., when iterating over the generator).
*   **Memory Usage:**  Extremely memory efficient, especially for large datasets, because it only stores the rule for generating values, not the values themselves.  It holds only one item in memory at a time.
*   **Return Value:** A generator object (an iterator).
*   **Iteration:**  You can iterate over a generator only *once*. After you've iterated through all its values, it's exhausted, and you need to create a new generator if you want to go through the values again.
*   **Example:**

    ```python
    numbers = [1, 2, 3, 4, 5]
    squares = (x*x for x in numbers)  # squares is a generator object
    for square in squares:
        print(square)
    ```

**Key Differences Summarized**

| Feature           | List Comprehension          | Generator Expression          |
| ----------------- | --------------------------- | ----------------------------- |
| Syntax            | `[expression for item in iterable if condition]` | `(expression for item in iterable if condition]` |
| Evaluation        | Eager (immediate)           | Lazy (on demand)              |
| Memory Usage      | Higher                      | Lower                       |
| Return Value      | List                        | Generator (iterator)           |
| Iteration         | Multiple times              | Single time only             |
| Stores all values? | Yes                         | No                             |

**When to Use Which**

1.  **Memory Constraints:**

    *   If you're dealing with a very large dataset or an infinite sequence of values, **use a generator expression**.  This will avoid loading everything into memory at once, preventing `MemoryError` exceptions and improving performance.
    *   If memory is not a concern and the dataset is relatively small, a list comprehension is fine.

2.  **Iteration Count:**

    *   If you need to iterate over the resulting sequence multiple times, **use a list comprehension**.  You can reuse the list as many times as you need.
    *   If you only need to iterate over the sequence once, **use a generator expression**.  This is often the case when passing the result to a function that consumes an iterator.

3.  **Performance:**

    *   If you need all the values available immediately and will be accessing them randomly, a list comprehension might be slightly faster because the values are already computed and stored.
    *   If you are only going to use a few of the values a generator expression can be much faster because you're only computing what you need.

4.  **Code Readability:**

    *   For simple transformations, list comprehensions can be more readable.
    *   For more complex transformations or situations requiring lazy evaluation, generator expressions can be more appropriate.

5. **Intermediate values**
    * If you need to store the resulting sequence for later use, use list comprehension.

**Illustrative Examples**

*   **Reading a large file line by line (generator expression):**

    ```python
    def process_file(filename):
        with open(filename, 'r') as f:
            # Generator expression to read and process each line
            lines = (line.strip() for line in f)  # Remove leading/trailing whitespace
            for line in lines:
                # Process the line
                print(line)
    ```

    In this example, you don't want to load the entire file into memory. The generator expression reads and processes lines one at a time.

*   **Creating a list of even numbers (list comprehension):**

    ```python
    even_numbers = [x for x in range(10) if x % 2 == 0]
    print(even_numbers)  # Output: [0, 2, 4, 6, 8]
    ```

    This is a simple and readable way to create a small list of even numbers. Because the range is small, memory is not a concern, and list comprehensions work great.

*   **Calculating the sum of squares of a large range (generator expression):**

    ```python
    def sum_of_squares(n):
        # Use a generator expression to calculate squares lazily and sum them
        squares = (x*x for x in range(n))
        return sum(squares)

    result = sum_of_squares(10**6)  # Large range
    print(result)
    ```

    Here, calculating the squares of a million numbers and storing them in a list would consume a lot of memory. A generator expression calculates each square only when the `sum()` function requests it, keeping memory usage low.

**In Summary**

*   Choose list comprehensions when you need a list, have enough memory, and might need to iterate multiple times.
*   Choose generator expressions when you need to process large datasets, want to conserve memory, and only need to iterate once.
*   Consider readability when choosing between the two, but prioritize memory efficiency when dealing with large amounts of data.

I hope this detailed explanation clarifies the differences between list comprehensions and generator expressions and helps you make informed decisions about when to use each one.  Let me know if you have any further questions!
